# Fase 4 — políticas com RAG e catálogo por consulta estruturada

O notebook importa os mesmos módulos `.py` do atendimento. Por padrão usa fixtures identificadas, sem OpenAI. A execução real e suas falhas/correções ficam no relatório separado. A fonte de políticas foi preservada: troca/garantia continuam pendentes.

Conceitos: chunking por parágrafo, embeddings, índice FAISS previamente preparado, retrieval, decomposição de consulta e geração no agente; text-to-query por plano Pydantic compilado em SELECT parametrizado. Não há SQL livre, indexação de políticas a cada pergunta ou associação de memória do cliente a regra comercial.

In [1]:
# AMBIENTE: localizar a cópia pelo marcador, sem depender do nome da pasta externa.
from pathlib import Path
import os
import sys
inicio=Path.cwd().resolve()
projeto=next(p for p in (inicio,*inicio.parents) if (p/'INSTRUCOES_PARA_IA_PROGRAMADORA.md').exists())
sys.path.insert(0,str(projeto/'02_codigo/src'))
modo=os.environ.get('FASE04_MODO','simulado')
print('Python:',sys.executable)
print('Modo:',modo)


Python: C:\Users\Renan\Documents\GIT_PROJETOS\estudos-engenharia-ia\.venv\Scripts\python.exe
Modo: simulado


## Recuperação e transformação de consulta

A busca inicial top-1 trouxe só entrega para a pergunta sobre cartão/parcelamento e frete. A correção decompõe assuntos e recupera trechos por consulta, eliminando IDs repetidos. O depois possui orçamento maior de recuperação; não atribuímos todo ganho exclusivamente à redação da consulta.

No modo real a preparação é feita antes, por `preparar_politicas.py --real`. Ao atender, `PoliticasRAG` carrega o arquivo existente e verifica fonte/modelo/hash. No modo simulado o exemplo cria um índice fixture numa pasta temporária, separado do índice de produção.

In [2]:
# RETRIEVAL/TEXT-TO-QUERY: o módulo compartilha as mesmas funções do atendimento.
from auto_super_parts.demonstracao_rag import demonstrar
r=demonstrar(modo)
print('Origem:',r['origem_embedding'])
print('Pergunta:',r['pergunta'])
print('Antes:',[d['id'] for d in r['antes']])
print('Depois:',[d['id'] for d in r['depois']['documentos']])
print('Consultas executadas:',r['depois']['consultas_executadas'])


Origem: fixture local identificada, sem API
Pergunta: Posso pagar com cartão em três vezes e tenho frete grátis em Curitiba?
Antes: ['POL-02']
Depois: ['POL-02', 'POL-01']
Consultas executadas: ['Posso pagar com cartão em três vezes e tenho frete grátis em Curitiba?', 'pagamento cartão parcelamento', 'entrega frete Curitiba']


In [3]:
# FONTES: inspecionar os trechos recuperados antes de julgar a resposta.
for d in r['depois']['documentos']:
    print()
    print(d['id'],d['secao'],d['fonte'],d['estado'])
    print(d['texto'])



POL-02 Entrega politicas_loja.txt documentado
Entrega: Curitiba/PR tem frete grátis; prazo local ainda não definido. Fora de Curitiba, transportadora com valor proporcional de R$ 8,85 por 200 km usando tabela de distâncias simuladas. Prazo externo de dois dias úteis por faixa de 100 km, arredondando para cima. Marco inicial ainda não definido. Cidade fora da tabela exige consulta ao atendimento.

POL-01 Pagamento politicas_loja.txt documentado
Pagamento: site somente Pix antecipado. Balcão Pix ou dinheiro, inclusive retirada. Sem cartão ou parcelamento. Entrega exige pagamento antecipado pelo site; não pagar ao entregador.


## Consulta estruturada restrita

A LLM real pode traduzir linguagem natural para `PlanoCatalogo`. Um compilador Python permite somente filtros/ordenações definidos e SELECT com parâmetros; o banco permanece somente leitura. O fixture deste exemplo fornece um plano pronto, não demonstra extração por LLM.

Listar velas por preço/estoque não exige veículo nem confirma aplicação no carro. Se a pergunta exigir aplicação e a configuração estiver incompleta, o planejador deve pedir esclarecimento. Dados comerciais continuam didáticos.

In [4]:
# PLANO E SQL: a execução real usa o contrato; nenhuma string SQL vem da LLM.
consulta=r['consulta_catalogo']
print('Status:',consulta['status'])
print('Plano:',consulta.get('plano'))
print('SQL:',consulta.get('sql_executado'))
print('Parâmetros:',consulta.get('parametros_sql'))
for p in consulta['produtos']:
    print(p['id'],p['categoria'],p['preco_centavos'],p['estoque'])
assert consulta['status']=='ok'
assert all(p['categoria']=='vela' and p['estoque']>0 and p['preco_centavos']<=2000 for p in consulta['produtos'])
assert {'POL-01','POL-02'} <= {d['id'] for d in r['depois']['documentos']}


Status: ok
Plano: {'consulta_valida': True, 'veiculo_id': None, 'categoria': 'vela', 'estoque': 'disponivel', 'preco_max_centavos': 2000, 'ordenacao': 'preco_crescente', 'limite': 10, 'pendencias': []}
SQL: SELECT DISTINCT p.dados FROM produtos p WHERE p.categoria=? AND p.estoque>0 AND p.preco_centavos<=? ORDER BY p.preco_centavos ASC, p.id LIMIT ?
Parâmetros: ['vela', 2000, 10]
P003 vela 1850 16
P009 vela 1850 16
P015 vela 1850 16
P021 vela 1850 16


## Atendimento integrado e verificações

```powershell
python -B 02_codigo/preparar_politicas.py --real
python -B 02_codigo/src/auto_super_parts/atendimento_rag.py --cliente cliente_demo
python -B 02_codigo/validar_rag.py --real
```

O índice real entregue pode ser carregado sem reconstrução; só o prepare de novo se a fonte/modelo mudar. Pergunte sobre pagamento ao entregador, cartão e frete em Curitiba, garantia, uma lista de velas até R$ 20,00 e o horário não documentado. O atendimento usa a memória já construída e acrescenta duas ferramentas. Para políticas explícitas, o middleware exige consulta no turno atual. O rodapé é montado com as fontes realmente recuperadas e persistido com a resposta.

As evidências reais preservam também a recusa indevida do planejador, a falta de consulta num turno e o falso positivo da avaliação inicial de listagem. Uma nota positiva ou chamada de ferramenta sozinha não comprova acerto. O verificador semântico geral/NLI e LangSmith ainda são próximos incrementos.